In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr
from scipy.spatial.distance import euclidean, cdist

import sys

sys.path.append("..")

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from skimage.filters import gaussian, median

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20251112_151422.log
/tmp/ipykernel_187408/4157130849.py:33: DeprecationWarning: PlottingFunctions.Plotter is deprecated and will be removed in a future version. Use PlottingBase.PublicationPlotter directly instead.
  plotter = PlottingFunctions.Plotter()


In [2]:
# data_folder = '/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/old/'
data_folder = "../Camera_Calibrations/Ximea_Camera/"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])
image_size = 12
camera_parameters = {}
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["variance"] = np.full((image_size, image_size), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size, image_size), np.median(readnoise))
camera_parameters["rqe"] = np.full((image_size, image_size), np.median(rqe))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ["B", "G", "R"]
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks

In [3]:
import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [33]:
notch_filter = "semrock-nf03-405-488-561-635e"
dichroic_mirror = "semrock-di03-r405-488-561-635-t1-25x36"
nilered_filter = "semrock-ff01-650-200-25"
shortpass_filter = "semrock-bsp01-785r"
dichroic_nored_mirror = "semrock-di03-r488-561-t1-25x36"
lp_561 = "semrock-blp02-561r"
#bp_584 = "semrock-ff01-582-64"

filters = [dichroic_nored_mirror, lp_561]
filter_spectra = S_F.get_dye_or_filter_data(
    names=filters, wavelength=wavelength, dye_or_filter=False
)

In [34]:
filter_spectra

array([[  2.74999999e-03,   3.56999994e-03,   1.35300001e-02, ...,
          9.70229983e-01,   9.70089972e-01,   9.71920013e-01],
       [  1.46130191e-07,   1.33928495e-07,   9.73773524e-08, ...,
          0.00000000e+00,   0.00000000e+00,   0.00000000e+00]])

In [35]:
S_F.get_pixel_fractions_dye_and_filters(filters=filters, dyes=['ATTO 565', 'ATTO 594'], pixel_QYs=pixel_QYs, wavelength=wavelength)

(array([ 618.05316526,  645.95819284]),
 array([[ 0.02833065,  0.33048532,  0.64118402],
        [ 0.03880403,  0.23301133,  0.72818465]]))

In [51]:
sigma = 1e-9
varepsilon = (sigma*6.023e23)/(np.log(10)*1e3)

In [56]:
varepsilon/1e11

2.6157556645032858

In [5]:
single_molecule_dyes = np.array(
    [
        ["ATTO 488", 2073],
        ["Alexa Fluor 488", 2811],
        ["CF488A", 3879],
        #["ATTO 594", 2000],
        ["Cy3B", 23195],
        ["ATTO 565", 11600],
        ["Janelia Fluor JF585-HaloTag conjugate", 2429],
        ["CF568", 13388],
        ["Cy5", 7801],
        ["Cy5B", 17090],
        ["ATTO 643", 23327],
        ["ATTO 647N", 18448],
        ["ATTO 655", 8273],
        ["CF640R", 12024],
        ["CF660R", 10399],
        ["abberior STAR 635", 12731],
        ["Janelia Fluor JF646-HaloTag conjugate", 14440],
        ["Alexa Fluor 647", 10348],
        ["Cy2", 6241],
        ["Cy3", 11022],
        ["Tetramethylrhodamine (TAMRA, TRITC)", 4884],
        ["Cy3.5", 4968],
        ["ATTO 647", 1526],
        ["ATTO 680", 1656],
        ["Cy5.5", 6337],
        ["Cy7", 852],
        ["Alexa Fluor 750", 703],
        ["ATTO 740", 779],
        ["Alexa Fluor 790", 740],
    ],
    dtype="object",
)

In [11]:
potential_dyes = [
    "Cy3B",
    #"ATTO 594",
    "CF550R",
    "Cy5B",
    "Alexa Fluor 488",
    "ATTO 488",
    "ATTO 643",
    "ATTO 647N",
    "ATTO 655",
    "Alexa Fluor 647",
    "CF640R",
    "CF660R",
    "abberior STAR 635",
    "ATTO 620",
    "ATTO 565",
    "CF568",
    "Janelia Fluor JF646-HaloTag conjugate",
    "ATTO 680",
    
]

In [14]:
import numpy as np
from src import Multicolour_Simulation_Functions
from src import SpectralFunctions

# Select optimal 5 dyes
result = MSF.optimal_dye_selector_simulated(
    potential_dyes=potential_dyes,
    single_molecule_dyes=single_molecule_dyes,
    filters=filters,
    smoothing_function=smoothing_function,
    camera_parameters=camera_parameters,
    wavelength=wavelength,
    n_dyes_desired=5,
    min_photons_per_100ms=500,
    n_simulations=10000,
    exhaustive_search=True,  # Use greedy for speed
    background_photons=50,
    verbose=True
)

print(f"\nOptimal dyes: {result['selected_dyes']}")
print(f"Classification accuracy: {result['overall_accuracy']:.1%}")


OPTIMAL DYE SELECTION VIA SIMULATION

Step 1: Filtering dyes (min 500 photons/100ms)...


  17 candidates -> 14 viable dyes
  Rejected: {'ATTO 620', 'ATTO 680', 'CF550R'}

Step 2-3: Simulating 10000 molecules per dye...
  Simulating Cy3B (23195 source / 4878 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.554, 0.413)
    Std  (A_R, A_G): (0.008, 0.008)
  Simulating Cy5B (17090 source / 3594 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.710, 0.222)
    Std  (A_R, A_G): (0.009, 0.008)
  Simulating Alexa Fluor 488 (2811 source / 591 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.090, 0.745)
    Std  (A_R, A_G): (0.019, 0.025)
  Simulating ATTO 488 (2073 source / 436 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.100, 0.745)
    Std

# Visualization of Optimal Dye Selection Process

## Plot 1: Exemplar Simulations for Each Selected Dye
Shows example fitted color values (A_R, A_G) from simulations of individual molecules.

In [ ]:
# Plot 1: Exemplar simulations from each selected dye
from PlottingBase import PublicationPlotter

pub_plotter = PublicationPlotter()
n_dyes = len(result["selected_dyes"])

fig, axes = plt.subplots(1, n_dyes, figsize=(4*n_dyes, 3.5))
if n_dyes == 1:
    axes = [axes]

for i, dye_name in enumerate(result["selected_dyes"]):
    ax = axes[i]
    
    # Get simulation data for this dye
    sim_data = result["dye_simulations"][dye_name]
    A_R = sim_data["A_R"]
    A_G = sim_data["A_G"]
    
    # Plot first 1000 examples
    n_plot = min(1000, len(A_R))
    ax.scatter(A_R[:n_plot], A_G[:n_plot], s=2, alpha=0.3, c="steelblue", rasterized=True)
    
    # Overlay Gaussian fit
    gauss = result["dye_gaussians"][dye_name]
    mean_R, mean_G = gauss["mean"]
    std_R, std_G = gauss["std_A_R"], gauss["std_A_G"]
    
    # Draw 2-sigma ellipse
    from matplotlib.patches import Ellipse
    ellipse = Ellipse((mean_R, mean_G), 2*2*std_R, 2*2*std_G, 
                     facecolor="none", edgecolor="red", linewidth=2, label="2$\sigma$ fit")
    ax.add_patch(ellipse)
    
    ax.set_xlabel("$ (Red Fraction)", fontsize=11)
    ax.set_ylabel("$ (Green Fraction)", fontsize=11)
    ax.set_title(f"{dye_name}
({result["expected_photons"][dye_name]:.0f} photons)", 
                fontsize=12, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=9, loc="upper right")

plt.tight_layout()
plt.show()


## Plot 2: Ternary Plot of All Simulated Localizations
Shows the complete color space with all viable dyes (uses datashader for large datasets).

In [ ]:
# Plot 2: Ternary plot of ALL localizations from all viable dyes
import datashader as ds
import datashader.transfer_functions as tf
import pandas as pd

pub_plotter = PublicationPlotter()
fig, ax = pub_plotter.create_figure(figsize=(8, 7))

# Collect all simulation data
all_A_R = []
all_A_G = []
all_dye_labels = []

for dye_name in result["dye_simulations"].keys():
    sim_data = result["dye_simulations"][dye_name]
    all_A_R.extend(sim_data["A_R"])
    all_A_G.extend(sim_data["A_G"])
    all_dye_labels.extend([dye_name] * len(sim_data["A_R"]))

print(f"Total localizations to plot: {len(all_A_R):,}")

# Use datashader for large datasets
df = pd.DataFrame({"A_R": all_A_R, "A_G": all_A_G})
canvas = ds.Canvas(plot_width=800, plot_height=800, 
                  x_range=(0, 1), y_range=(0, 1))
agg = canvas.points(df, "A_R", "A_G")
img = tf.shade(agg, cmap=["lightblue", "darkblue"], how="log")
img_array = img.to_pil()

ax.imshow(img_array, extent=[0, 1, 0, 1], origin="lower", aspect="auto")
ax.set_xlabel("$ (Red Fraction)", fontsize=14)
ax.set_ylabel("$ (Green Fraction)", fontsize=14)
ax.set_title("All Simulated Dye Color Distributions
(Datashader Density Plot)", 
            fontsize=15, fontweight="bold")
ax.grid(True, alpha=0.3, color="white", linewidth=0.5)

plt.tight_layout()
plt.show()


## Plot 3: Combination Testing Process
Shows how different dye combinations were tested and highlights on the ternary plot.

In [ ]:
# Plot 3: Iterative combination testing
if result["all_combinations_tested"] is not None:
    # Exhaustive search mode - show top 10 combinations
    combos = result["all_combinations_tested"][:10]
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.ravel()
    
    for i, combo_result in enumerate(combos):
        ax = axes[i]
        combo_dyes = combo_result["dyes"]
        accuracy = combo_result["accuracy"]
        
        # Plot background (all dyes, faint)
        for dye_name in result["dye_simulations"].keys():
            sim_data = result["dye_simulations"][dye_name]
            n_plot = min(500, len(sim_data["A_R"]))
            ax.scatter(sim_data["A_R"][:n_plot], sim_data["A_G"][:n_plot], 
                      s=1, alpha=0.05, c="gray", rasterized=True)
        
        # Highlight selected combination
        colors = plt.cm.tab10(range(len(combo_dyes)))
        for j, dye_name in enumerate(combo_dyes):
            if dye_name in result["dye_simulations"]:
                sim_data = result["dye_simulations"][dye_name]
                n_plot = min(200, len(sim_data["A_R"]))
                ax.scatter(sim_data["A_R"][:n_plot], sim_data["A_G"][:n_plot],
                          s=10, alpha=0.6, c=[colors[j]], label=dye_name[:15], rasterized=True)
        
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("$", fontsize=9)
        ax.set_ylabel("$", fontsize=9)
        
        rank_str = "BEST" if i == 0 else f"#{i+1}"
        ax.set_title(f"{rank_str}: Accuracy = {accuracy:.1%}", 
                    fontsize=10, fontweight="bold" if i == 0 else "normal")
        ax.legend(fontsize=7, loc="upper right", framealpha=0.8)
        ax.grid(True, alpha=0.2)
    
    plt.suptitle("Top 10 Dye Combinations (Exhaustive Search)", 
                fontsize=16, fontweight="bold", y=1.00)
    plt.tight_layout()
    plt.show()
else:
    print("Greedy search mode - no combination history available")
    print("Rerun with exhaustive_search=True to visualize all tested combinations")


## Plot 4: Optimal Dye Selection
Shows the final selected dyes and their separation in color space.

In [ ]:
# Plot 4: Optimal dye selection with overlap visualization
from matplotlib.patches import Ellipse
from scipy.stats import multivariate_normal

pub_plotter = PublicationPlotter()
fig, ax = pub_plotter.create_figure(figsize=(10, 9))

selected_dyes = result["selected_dyes"]
colors = plt.cm.tab10(range(len(selected_dyes)))

# Plot each selected dye
for i, dye_name in enumerate(selected_dyes):
    sim_data = result["dye_simulations"][dye_name]
    gauss = result["dye_gaussians"][dye_name]
    
    # Plot simulation points
    n_plot = min(1000, len(sim_data["A_R"]))
    ax.scatter(sim_data["A_R"][:n_plot], sim_data["A_G"][:n_plot],
              s=8, alpha=0.4, c=[colors[i]], label=dye_name, rasterized=True)
    
    # Draw 2-sigma ellipse
    mean_R, mean_G = gauss["mean"]
    std_R, std_G = gauss["std_A_R"], gauss["std_A_G"]
    ellipse = Ellipse((mean_R, mean_G), 2*2*std_R, 2*2*std_G,
                     facecolor="none", edgecolor=colors[i], linewidth=2.5)
    ax.add_patch(ellipse)
    
    # Annotate with dye name
    ax.text(mean_R, mean_G, str(i+1), fontsize=14, fontweight="bold",
           ha="center", va="center", color="white",
           bbox=dict(boxstyle="circle", facecolor=colors[i], edgecolor="white", linewidth=2))

ax.set_xlabel("$ (Red Fraction)", fontsize=14)
ax.set_ylabel("$ (Green Fraction)", fontsize=14)
ax.set_title(f"Optimal {len(selected_dyes)}-Dye Combination
"
            f"Overall Accuracy: {result["overall_accuracy"]:.1%}",
            fontsize=16, fontweight="bold")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=11, loc="upper right", framealpha=0.9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Plot 5: Classification Accuracy Analysis
Shows confusion matrix and pairwise separation statistics.

In [ ]:
# Plot 5: Classification accuracy determination
from PlottingBase import PublicationPlotter

pub_plotter = PublicationPlotter()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Confusion Matrix
ax = axes[0]
confusion = result["confusion_matrix"]
selected_dyes = result["selected_dyes"]

im = ax.imshow(confusion, cmap="Blues", vmin=0, vmax=1, aspect="auto")

# Add text annotations
for i in range(len(selected_dyes)):
    for j in range(len(selected_dyes)):
        value = confusion[i, j]
        color = "white" if value > 0.5 else "black"
        ax.text(j, i, f"{value:.3f}", ha="center", va="center",
               fontsize=10, fontweight="bold", color=color)

ax.set_xticks(range(len(selected_dyes)))
ax.set_yticks(range(len(selected_dyes)))
ax.set_xticklabels([f"{i+1}" for i in range(len(selected_dyes))], fontsize=11)
ax.set_yticklabels([f"{i+1}" for i in range(len(selected_dyes))], fontsize=11)
ax.set_xlabel("Predicted Dye", fontsize=13, fontweight="bold")
ax.set_ylabel("True Dye", fontsize=13, fontweight="bold")
ax.set_title("Confusion Matrix
(Monte Carlo Classification)", 
            fontsize=14, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Classification Probability", fontsize=11)

# Panel B: Per-Dye Accuracy
ax = axes[1]
per_dye_accuracy = np.diag(confusion)
dye_labels = [f"{i+1}. {dye[:20]}" for i, dye in enumerate(selected_dyes)]

bars = ax.barh(range(len(selected_dyes)), per_dye_accuracy, 
              color=plt.cm.tab10(range(len(selected_dyes))), alpha=0.7)
ax.set_yticks(range(len(selected_dyes)))
ax.set_yticklabels(dye_labels, fontsize=10)
ax.set_xlabel("Classification Accuracy", fontsize=13, fontweight="bold")
ax.set_title("Per-Dye Classification Accuracy", fontsize=14, fontweight="bold")
ax.set_xlim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis="x")

# Add value labels
for i, (bar, acc) in enumerate(zip(bars, per_dye_accuracy)):
    ax.text(acc - 0.002, bar.get_y() + bar.get_height()/2, 
           f"{acc:.1%}", ha="right", va="center", 
           fontsize=11, fontweight="bold", color="white")

plt.tight_layout()
plt.show()

# Print summary statistics
print("
" + "="*60)
print("CLASSIFICATION ACCURACY SUMMARY")
print("="*60)
for i, dye in enumerate(selected_dyes):
    acc = per_dye_accuracy[i]
    print(f"{i+1}. {dye:40s}: {acc:.2%}")
print(f"
Overall accuracy: {result["overall_accuracy"]:.2%}")
print("="*60)
